In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import math 

# 1. Feature Engineering Engine

In [2]:
def create_features(df):
    data = df.copy()
    
    # Interaction terms
    data['risk_risk_mult'] = data['foreign_transaction'] * data['location_mismatch']
    
    # Hour cyclical encoding
    data['hour_sin'] = np.sin(2 * np.pi * data['transaction_hour'] / 24.0)
    data['hour_cos'] = np.cos(2 * np.pi * data['transaction_hour'] / 24.0)
    
    # Categorical One-Hot Encoding
    data = pd.get_dummies(data, columns=['merchant_category'])
    return data

# Load Data

In [3]:
train = pd.read_csv('/kaggle/input/competitions/oct-wave-3-0-credit-card-fraud-detection-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/oct-wave-3-0-credit-card-fraud-detection-challenge/test.csv')
print(train)
print(test)
y_train = train['is_fraud'].values
X_train_raw = train.drop(columns=['is_fraud', 'transaction_id'])
X_test_raw = test.drop(columns=['transaction_id'])


      transaction_id  amount  transaction_hour merchant_category  \
0               6403   40.96                 6              Food   
1               5803   57.07                 6       Electronics   
2               4132   44.24                 0           Grocery   
3               7627  396.80                23          Clothing   
4               6047   20.13                 0              Food   
...              ...     ...               ...               ...   
7995            4722   10.13                20          Clothing   
7996            3699  487.10                19            Travel   
7997             861  268.92                 5           Grocery   
7998            7360  374.82                15              Food   
7999            9892   15.85                 8          Clothing   

      foreign_transaction  location_mismatch  device_trust_score  \
0                       0                  1                  82   
1                       0                  0   

# Prepare Features

In [4]:
X_train = create_features(X_train_raw)
X_test = create_features(X_test_raw)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

# 2. Out-of-Fold Cross Validation Setup

In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
    
    # Model 1: LightGBM
    m1 = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.1, num_leaves=31, 
                            min_child_samples=5, scale_pos_weight=scale_pos_weight, random_state=42, verbosity=-1)
    m1.fit(X_tr, y_tr)
    
    # Model 2: XGBoost
    m2 = xgb.XGBClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, 
                           scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')
    m2.fit(X_tr, y_tr)
    
    # Model 3: CatBoost
    m3 = cb.CatBoostClassifier(iterations=200, learning_rate=0.08, depth=5, 
                               scale_pos_weight=scale_pos_weight, random_state=42, verbose=0)
    m3.fit(X_tr, y_tr)
    
    # Weighted Average Predictions
    val_pred = 0.5 * m1.predict_proba(X_va)[:, 1] + 0.3 * m3.predict_proba(X_va)[:, 1] + 0.2 * m2.predict_proba(X_va)[:, 1]
    oof_preds[val_idx] = val_pred
    
    test_pred = 0.5 * m1.predict_proba(X_test)[:, 1] + 0.3 * m3.predict_proba(X_test)[:, 1] + 0.2 * m2.predict_proba(X_test)[:, 1]
    test_preds += test_pred / skf.n_splits

# 3. Find Global Optimal Threshold

In [6]:
best_t, best_f1 = 0.5, 0.0
for t in np.linspace(0.01, 0.99, 200):
    score = f1_score(y_train, (oof_preds >= t).astype(int))
    if score > best_f1:
        best_f1, best_t = score, t

print(f"Optimal Threshold: {best_t:.3f} | Out-of-Fold F1-Score: {best_f1:.4f}")

Optimal Threshold: 0.192 | Out-of-Fold F1-Score: 0.9917


# 4. Generate Final Winning Submission

In [7]:
final_binary_preds = (test_preds >= best_t).astype(int)
submission = pd.DataFrame({'transaction_id': test['transaction_id'], 'is_fraud': final_binary_preds})
submission.to_csv('submission.csv', index=False)
print(submission)

      transaction_id  is_fraud
0               9945         0
1               2602         0
2                343         0
3               3223         0
4               5217         0
...              ...       ...
1995            9419         0
1996            8903         0
1997            7239         0
1998            2965         0
1999            5854         0

[2000 rows x 2 columns]
